In [ ]:
!pip install -U transformers vllm math-verify

In [ ]:
import sys
import torch
import transformers
import vllm

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)
print("GPUs:", torch.cuda.device_count())

In [ ]:
import json
import glob
from pathlib import Path

dataset_path=glob.glob('/kaggle/input/**/review_pairs.json',recursive=True)[0]
print(dataset_path)

with open(dataset_path,'r',encoding='utf-8') as file:
    pairs=json.load(file)

print(len(pairs))

In [ ]:
from collections import Counter
Counter(p['type'] for p in pairs)

In [ ]:
def format_teacher_prompt(record: dict)->str:
    result=""

    result+='Below is a math problem and two attempted solutions written by a student. '
    result+='The correct final answer is: '+record.get('gold')+'\n\n'

    result+='[Problem]\n'+record.get('problem')+'\n\n'
    result+='[Answer 1]\n'+record.get('answer_1')+'\n\n'
    result+='[Answer 2]\n'+record.get('answer_2')+'\n\n'

    result+='Write a short review (3 to 6 sentences) in the voice of the student re-checking their own two attempts.\n'
    result+='- Say which final answer each attempt reached.\n'
    result+='- If they disagree, point to the exact step where the wrong attempt makes its mistake and explain the error briefly.\n'
    result+='- If they agree, briefly confirm the key step that makes the answer right.\n'
    result+='- Do not mention that you were told the correct answer.\n'
    result+='- End with exactly: Therefore the final answer is \\boxed{...}.'

    return result

In [ ]:
print(format_teacher_prompt(pairs[0]))

In [ ]:
from vllm import LLM, SamplingParams

teacher_name='Qwen/Qwen3.5-9B'

llm=LLM(
    model=teacher_name,
    dtype="float16",
    tensor_parallel_size=2,
    max_model_len=6144,
    gpu_memory_utilization=0.90,
    limit_mm_per_prompt={"image": 0, "video": 0},
)

print("Teacher loaded")

In [ ]:
review_params=SamplingParams(
    n=2,
    temperature=0.3,
    top_p=0.95,
    max_tokens=512,
    seed=0,
)

In [ ]:
messages=[[{'role': 'user', 'content': format_teacher_prompt(i)}] for i in pairs]

outputs=llm.chat(
    messages,
    review_params,
    chat_template_kwargs={"enable_thinking": False},
)

print("Reviews generated")

In [ ]:
def extract_boxed(text: str):
    start=text.rfind('\\boxed{')
    if start==-1:
        return None

    i=start+len('\\boxed{')
    depth=1
    while i<len(text) and depth>0:
        if text[i]=='{':
            depth+=1
        elif text[i]=='}':
            depth-=1
        i+=1

    if depth!=0:
        return None
    return text[start+len('\\boxed{'):i-1]

In [ ]:
from math_verify import parse, verify

def is_correct(text: str, gold: str)->bool:
    try:
        return bool(verify(parse('$'+gold+'$'),parse(text)))
    except Exception:
        return False

In [ ]:
leak_words=['correct final answer is', 'told the correct', 'given answer', 'reference answer', 'was given']

def is_good_review(text: str, gold: str)->bool:
    if extract_boxed(text) is None:
        return False
    if not is_correct(text,gold):
        return False
    if any(w in text.lower() for w in leak_words):
        return False
    if len(text)>2000:
        return False
    return True

In [ ]:
for record,output in zip(pairs,outputs):
    record['review']=None
    for j in output.outputs:
        if is_good_review(j.text,record['gold']):
            record['review']=j.text.strip()
            break

kept=[p for p in pairs if p['review']!=None]

print("Kept:",len(kept),"out of",len(pairs))
print(Counter(p['type'] for p in kept))

In [ ]:
import random

for p in random.sample(kept,3):
    print("="*80)
    print(p['type'],'| gold:',p['gold'],'| pred_1:',p['pred_1'],'| pred_2:',p['pred_2'])
    print(p['review'])

In [ ]:
savepath=Path('/kaggle/working/reviews.json')

with savepath.open('w',encoding='utf-8') as file:
    json.dump(kept,file,ensure_ascii=False,indent=2)

print("Saved:",savepath)